# Architecture et entraînement

## Fil rouge

Les données sont prêtes. Troisième étape : construire et entraîner le modèle. Vous allez créer un petit réseau de neurones qui apprend à prédire le token suivant.

## Contexte

Vous avez des tokens. Vous voulez un modèle qui, given une séquence de tokens, prédit le token suivant. Le modèle est composé de :

1. **Embeddings** : chaque token est converti en vecteur de dimension fixe
2. **Couches** : transformations (linéaires, activation) pour capturer les patterns
3. **Head de prédiction** : sortie de dimension vocabulaire (probabilité pour chaque token)

## Ce qu'on attend de vous

- Comprendre les embeddings, les couches et la loss
- Écrire la boucle d'entraînement
- Avoir entraîné un modèle et l'avoir sauvegardé

## Comment faire

1. Définissez un modèle simple (embedding + MLP ou couche linéaire)
2. Préparez les données (paires input/target)
3. Boucle d'entraînement : forward, loss (cross-entropy), backward, step
4. Sauvegardez le modèle à la fin

In [2]:
import torch
import torch.nn as nn

# Chargez les données et le vocabulaire (réutilisez le notebook 02)
with open('sample_text.txt', 'r', encoding='utf-8') as f:
    texte = f.read()

caracteres_uniques = sorted(set(texte))
char_to_id = {c: i for i, c in enumerate(caracteres_uniques)}
vocab_size = len(char_to_id)

# Convertir le texte en tenseur
data = torch.tensor([char_to_id[c] for c in texte], dtype=torch.long)

# A COMPLETER : Modèle simple (embedding + linéaire)
class SimpleLLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x : (batch, seq_len) -> logits : (batch, seq_len, vocab_size)
        h = self.embed(x)
        h = torch.relu(self.fc(h))
        return self.out(h)

model = SimpleLLM(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# A COMPLETER : Boucle d'entraînement
seq_len = 32  # Longueur de la fenêtre de contexte
for epoch in range(100):  # Ajustez le nombre d'epochs
    # Créez des batches (input = tokens 0..n-1, target = tokens 1..n)
    idx = torch.randint(0, len(data) - seq_len - 1, (64,))  # 64 exemples par batch
    x = torch.stack([data[i:i+seq_len] for i in idx])
    y = torch.stack([data[i+1:i+seq_len+1] for i in idx])
    
    logits = model(x)
    loss = nn.functional.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Sauvegardez le modèle
torch.save({'model': model.state_dict(), 'char_to_id': char_to_id}, 'model_simple.pt')

Epoch 0, Loss: 3.9368
Epoch 10, Loss: 3.4455
Epoch 20, Loss: 3.0360
Epoch 30, Loss: 2.7324
Epoch 40, Loss: 2.6130
Epoch 50, Loss: 2.5103
Epoch 60, Loss: 2.4356
Epoch 70, Loss: 2.3728
Epoch 80, Loss: 2.3393
Epoch 90, Loss: 2.2889


## À la fin de ce notebook, vous devez être capable de :

- Expliquer chaque composant du modèle (embedding, couches, head)
- Expliquer la boucle d'entraînement (forward, loss, backward, step)